# AgriSmart AI: Emergency 1-Hour GPU Training

This notebook trains the multi-crop disease detection model using Google Colab's T4/A100 GPU.

## Cell 1: Enable GPU verification

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is unavailable. In Colab, select Runtime > Change runtime type > T4 GPU.")

## Cell 2: Install dependencies

In [ ]:
!pip install torch torchvision pandas scikit-learn pillow

## Cell 3: Obtain the project
Upload a ZIP of your AgriSmart-AI project directory using the left sidebar in Colab, or clone your private repo. Below assumes you uploaded `AgriSmart-AI.zip`.

In [ ]:
!unzip -q AgriSmart-AI.zip -d .
%cd AgriSmart-AI

## Cell 4: Obtain the Dataset
Since the raw images (data/external/plantvillage) are ignored in Git, you must provide them. The easiest way is to download the HuggingFace PlantVillage ZIP directly to Colab and extract it into `data/external/plantvillage_raw/raw/color`.

In [ ]:
import os

# Check if dataset is already present
dataset_dir = "data/external/plantvillage"
if not os.path.exists(dataset_dir):
    print("Downloading PlantVillage dataset directly to Colab...")
    !wget -q https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/data.zip
    !unzip -q data.zip -d data/external/plantvillage_raw
    !mv data/external/plantvillage_raw/raw/color data/external/plantvillage
    !rm data.zip

print("Dataset ready at:", dataset_dir)

## Cell 5: Run Preflight Checks

In [ ]:
import json
import pandas as pd

with open("data/class_registry.json", "r") as f:
    registry = json.load(f)
    
train_df = pd.read_csv("data/manifests/train.csv")
val_df = pd.read_csv("data/manifests/val.csv")
test_df = pd.read_csv("data/manifests/test.csv")

print("Preflight Validation:")
print(f"- Classes in Registry: {len(registry)} (Expected: 38)")
print(f"- Train Set: {len(train_df)} (Expected: 37982)")
print(f"- Validation Set: {len(val_df)} (Expected: 8126)")
print(f"- Test Set: {len(test_df)} (Expected: 8176)")

t_hashes = set(train_df['file_hash'])
v_hashes = set(val_df['file_hash'])
te_hashes = set(test_df['file_hash'])

if t_hashes.intersection(v_hashes) or t_hashes.intersection(te_hashes) or v_hashes.intersection(te_hashes):
    raise ValueError("CRITICAL ERROR: Leakage detected!")
print("- Data Leakage Check: PASSED")

## Cell 6: Run Rapid Training

In [ ]:
!python model/train_colab_fast.py \
  --data-root data/external/plantvillage \
  --train-manifest data/manifests/train.csv \
  --val-manifest data/manifests/val.csv \
  --test-manifest data/manifests/test.csv \
  --output-dir model/artifacts/plant_disease_colab \
  --architecture efficientnet_b0 \
  --epochs 3 \
  --batch-size auto \
  --num-workers 2 \
  --max-training-minutes 40 \
  --seed 42

## Cell 7: Evaluate Selected Checkpoint

In [ ]:
import os
os.environ['TEST_CHECKPOINT'] = 'model/artifacts/plant_disease_colab/best_model.pth'
os.environ['EVALUATE_COLAB'] = '1'

# Ensure evaluate.py knows to load the colab checkpoint
!python model/evaluate.py

## Cell 8: Test Prediction

In [ ]:
test_image = train_df.iloc[0]['relative_path']
!python model/predict.py data/$test_image

## Cell 9: Download Artifacts
Run this cell to zip up the final trained model and metrics so you can download them.

In [ ]:
!zip -r colab_artifacts.zip model/artifacts/plant_disease_colab/
from google.colab import files
files.download('colab_artifacts.zip')